# 01 — EDA i kontrakt danych (Sprint 1)

Cel notebooka: poznać dane wybranego datasetu i podjąć decyzje, które trafią do kodu:

- **input normalization** — reguły w `normalize_input()` (`src/asi_project/data.py`),
- **training-only filtering** — reguły w `filter_training_rows()` (jeśli są potrzebne),
- **typy cech** — listy `features.numeric` i `features.categorical` w `project.yaml`,
- **metryka główna** — uzasadnienie w karcie projektu w README.

Jak pracować:

- uruchamiajcie notebook od góry (Restart & Run All); po zmianie `project.yaml` uruchomcie
  ponownie komórkę ustawień,
- komórki oznaczone **TODO** uzupełnia zespół; odpowiedzi tekstowe wpisujcie w miejsce
  `*(odpowiedź)*`,
- nie zmieniajcie kontraktu prowadzącej (`src/asi_project/datasets.yaml`).

## 0. Ustawienia

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from asi_project.config import get_contract, load_project_config
from asi_project.data import (
    encode_target,
    feature_list_problems,
    filter_training_rows,
    is_text_column,
    load_dataset,
    normalize_input,
    target_classes,
    validate,
)

config = load_project_config()
contract = get_contract(config)
DATA_MODE = os.environ.get("ASI_DATA_MODE", "full")

print(f"Dataset: {contract.title} ({contract.dataset_id})")
print(f"Target: {contract.target} | klasa pozytywna: {contract.positive_class}")
print(f"Moment predykcji: {contract.prediction_moment}")

## 1. Wczytanie danych

Pełne dane pobiera `python scripts/download_data.py`. Nazwy kolumn są ujednolicane przy
wczytywaniu (białe znaki), wartości — jeszcze nie.

In [ ]:
df_raw = load_dataset(config, contract, mode=DATA_MODE)
print(df_raw.shape)
df_raw.head()

## 2. Inspekcja: typy, braki, liczba unikalnych wartości

In [ ]:
overview = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "braki": df_raw.isna().sum(),
        "unikalne": df_raw.nunique(),
    },
)
overview


*Typy kolumn wyglądają poprawnie. Nie ma braków, ale są wartości unknown oraz -1 reprezentujące brak . Kolumnę day można rozważyć jako kategoryczną, ponieważ oznacza dzień miesiąca. pdays pozostawiamy jako cechę liczbową, ale wartość -1 traktujemy osobno.*

## 3. Kontrakt danych

Kontrakt prowadzącej określa target, klasę pozytywną, ALLOWED FEATURES (jedyne kolumny
dozwolone w X) i EXCLUDED FEATURES wraz z powodem wykluczenia.

In [ ]:
problems = validate(df_raw, contract)
print("Kontrakt danych:", "OK" if not problems else problems)
print("ALLOWED FEATURES:", list(contract.allowed_features))
print("EXCLUDED FEATURES:")
for column, reason in contract.excluded_features.items():
    status = "jest w danych" if column in df_raw.columns else "brak w pliku"
    print(f"- {column} ({status}): {reason}")

## 4. Target i klasa pozytywna

In [ ]:
print("Klasy targetu:", target_classes(df_raw[contract.target]))
y = encode_target(df_raw[contract.target].dropna(), contract.positive_class)
print(f"Udział klasy pozytywnej: {y.mean():.3f}")
ax = y.value_counts().sort_index().plot(kind="bar", title="Target (1 = klasa pozytywna)")
ax.set_xlabel("klasa")
plt.show()


*Jako główną metrykę wybieramy F1 dla klasy yes, ponieważ zależy nam zarówno na trafnym wskazywaniu zainteresowanych klientów, jak i na tym, żeby nie pomijać ich zbyt wielu. F1 łączy precision i recall, uwzględniając oba te cele. Sama accuracy byłaby myląca, ponieważ większość klientów nie zakłada lokaty, więc wysoki wynik można uzyskać, przewidując zawsze no.*

## 5. EDA


In [ ]:
display(df_raw.groupby(contract.target)["age"].describe())

age_groups = pd.cut(
    df_raw["age"],
    bins=[17, 29, 39, 49, 59, 100],
    labels=["18–29", "30–39", "40–49", "50–59", "60+"],
)

age_rates = pd.crosstab(age_groups, df_raw[contract.target], normalize="index")

(age_rates["yes"] * 100).plot(kind="bar", rot=0)
plt.xlabel("Wiek")
plt.ylabel("Klienci zakładający lokatę (%)")
plt.show()

In [ ]:
display(pd.crosstab(df_raw["job"], df_raw[contract.target], normalize="index"))

pd.crosstab(df_raw["job"], df_raw[contract.target]).plot(kind="bar")

In [ ]:
pd.DataFrame({"unknown": (df_raw == "unknown").sum(), "-1": (df_raw == -1).sum()})

In [ ]:
df_raw.loc[df_raw["balance"] < 0, "balance"].describe()

In [ ]:
df_raw.loc[df_raw["pdays"] < 0, "pdays"].describe()


*Sam średni wiek ukrywa istotną zależność: lokaty częściej zakładają klienci z najmłodszej i najstarszej grupy niż osoby w średnim wieku. Sugeruje to, że prosty liniowy wpływ wieku może nie wystarczyć do opisania tej zależności w modelu. Zawód również może pomagać w rozpoznawaniu zainteresowanych klientów, ale wysoki odsetek yes nie oznacza największej liczby potencjalnych klientów. Przy planowaniu kampanii warto więc uwzględniać zarówno odsetek zainteresowanych w danej grupie, jak i jej wielkość. Wartości unknown wskazują brak informacji w job, education, contact i poutcome, natomiast wartości ujemne mają różne znaczenie: w balance są rzeczywistymi saldami, a w pdays występuje wyłącznie -1, oznaczające brak wcześniejszego kontaktu.*

## 6. Input normalization

Input normalization działa na pojedynczym rekordzie, nie używa targetu ani statystyk zbioru
i jest idempotentna. Typowe kandydatki: białe znaki, puste napisy, oczywiste reprezentacje
braków, ujednolicenie zapisu wartości. Uzupełnianie braków medianą **nie jest** input
normalization — wymaga `fit`, więc należy do model preprocessing.

In [ ]:
text_columns = [column for column in df_raw.columns if is_text_column(df_raw[column])]
for column in text_columns:
    values = df_raw[column].dropna().astype(str)
    padded = int((values != values.str.strip()).sum())
    empty = int((values.str.strip() == "").sum())
    top = values.value_counts().head(6).to_dict()
    print(f"{column}: spacje na brzegach={padded}, puste={empty}, najczęstsze={top}")


*Dodajemy zamianę -1 w pdays na NaN, ponieważ oznacza ono brak wcześniejszego kontaktu, a nie rzeczywistą liczbę dni. Reguła jest bezpieczna, ponieważ nie korzysta z targetu ani statystyk zbioru, a ponowne wykonanie nie zmienia wyniku.*

In [ ]:
normalized = normalize_input(df_raw)
changed = (normalized.astype(str) != df_raw.astype(str)).sum()
changed[changed > 0]

## 7. Training-only filtering

Training-only filtering działa tylko przy przygotowaniu danych treningowych i może jedynie
usuwać rekordy. Domyślnie `filter_training_rows()` usuwa rekordy bez targetu.

In [ ]:
print("Rekordy bez targetu:", int(df_raw[contract.target].isna().sum()))
print("Dokładne duplikaty wierszy:", int(df_raw.duplicated().sum()))
filtered = filter_training_rows(normalize_input(df_raw), contract)
print(f"Po training-only filtering: {len(filtered)} z {len(df_raw)} rekordów")


*Nie usuwamy dodatkowych rekordów, bo nie ma duplikatów ani braków w targecie.*

## 8. Typy cech → `project.yaml`

**TODO:** wpiszcie do `project.yaml` listy `features.numeric` i `features.categorical`
(wyłącznie ALLOWED FEATURES), zapiszcie plik i uruchomcie ponownie komórkę ustawień oraz
komórkę poniżej.

In [ ]:
print("Numeryczne:", list(config.numeric_features))
print("Kategoryczne:", list(config.categorical_features))
print("Problemy z deklaracją:", feature_list_problems(contract, config.model_features) or "brak")
print("Kontrakt z deklaracją:", validate(normalize_input(df_raw), contract, config) or "OK")

## 9. Podsumowanie decyzji

*W normalizacji zamieniamy -1 w pdays na brak danych, a unknown zostawiamy jako osobną kategorię. Nie usuwamy dodatkowych rekordów, bo nie znaleźliśmy duplikatów ani braków w targecie. Zachowujemy ustalony podział na cechy liczbowe i kategoryczne, pozostawiając day i pdays jako liczbowe. Jako główną metrykę wybieramy F1 dla klasy yes, ponieważ uwzględnia zarówno trafność wskazań, jak i wykrywanie zainteresowanych klientów przy nierównych proporcjach klas.*